In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

In [0]:
df_raw = spark.read.table('ipl.s1_raw.ipl_data')
display(df_raw)

In [0]:
df_deliveries = df_raw.select(posexplode('innings').alias('innings_num','innings'),'match_id','team1','team2')
df_deliveries = df_deliveries\
                .withColumn('innings_num',df_deliveries.innings_num+lit(1))\
                .withColumn('innings_id',concat(df_deliveries.match_id,lit('i'),df_deliveries.innings_num))\
                .withColumn('batting_team',df_deliveries.innings.team)\
                .withColumn('fielding_team',when(df_deliveries.team1 == df_deliveries.batting_team,df_deliveries.team2).otherwise(df_deliveries.team1))\
                .drop('team1','team2')
            
df_deliveries = df_deliveries\
                .select(posexplode(df_deliveries.innings.overs).alias('over_num','overs_array'),'*')\
                

df_deliveries = df_deliveries\
                .select(posexplode(df_deliveries.overs_array.deliveries).alias('delivery_num','delivery_array'),'*')\
                .withColumn('delivery_num',(df_deliveries.over_num+(df_deliveries.delivery_num+lit(1))*.1).cast('decimal(5,1)'))\
                .withColumn('batter',df_deliveries.delivery_array.batter)\
                .withColumn('bowler',df_deliveries.delivery_array.bowler)\
                .withColumn('non_striker',df_deliveries.delivery_array.non_striker)\
                .withColumn('runs_tot',df_deliveries.delivery_array.runs.total)\
                .withColumn('runs_batter',df_deliveries.delivery_array.runs.batter)\
                .withColumn('runs_extras',df_deliveries.delivery_array.runs.extras)\
                .withColumn('over_id',concat(df_deliveries.innings_id,lit('o'),lpad(df_deliveries.over_num.cast('string'),2,0)))\
                .withColumn('delivery_id',concat(df_deliveries.over_id,lit('d'),(df_deliveries.delivery_num*10%10).cast('int').cast('string')))\
                .withColumn('wicket_flag',when(df_deliveries.delivery_array.wickets.isNotNull(),lit(1)).otherwise(lit(0)))\
                .withColumn(
                    'batter_out',
                    when(df_deliveries.delivery_array.wickets.isNotNull(),df_deliveries.delivery_array.wickets[0].player_out)
                    .otherwise(lit(''))
                    )\
                .withColumn(
                    'wicket_type',
                    when(df_deliveries.delivery_array.wickets.isNotNull(),df_deliveries.delivery_array.wickets[0].kind)
                    .otherwise(lit(''))
                    )\
                .withColumn(
                    'fielders',
                    when(df_deliveries.delivery_array.wickets[0].fielders.isNotNull(),
                         when(df_deliveries.delivery_array.wickets[0].kind == 'caught and bowled',df_deliveries.bowler)
                         .otherwise(df_deliveries.delivery_array.wickets[0].fielders[0].getField('name'))
                        )
                    .otherwise(lit(''))
                    )\
                
                

df_deliveries = df_deliveries.select\
                (
                    df_deliveries.match_id,
                    df_deliveries.innings_id,
                    df_deliveries.over_id,
                    df_deliveries.delivery_id,
                    df_deliveries.delivery_num,
                    df_deliveries.batter,
                    df_deliveries.non_striker,
                    df_deliveries.bowler,
                    df_deliveries.runs_batter,
                    df_deliveries.runs_extras,
                    df_deliveries.runs_tot,
                    df_deliveries.wicket_flag,
                    df_deliveries.batter_out,
                    df_deliveries.wicket_type,
                    df_deliveries.fielders,
                    df_deliveries.innings_num,
                    df_deliveries.over_num,
                    df_deliveries.batting_team,
                    df_deliveries.fielding_team,
                    df_deliveries.delivery_array
                ).drop("delivery_array")\
                .orderBy('delivery_id')

display(df_deliveries)

In [0]:
df_deliveries.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable('ipl.s2_silver.deliveries')